# Aprendizaje del E-puck: Modelo Predictivo (forward) + Modelo Inverso curiosidad-ponderado

**Qué cambia respecto a la versión anterior:**

1. **Modelo Predictivo (forward model)** — nuevo. Recibe el estado del robot en *t*
   (sensores + necesidades: batería y aburrimiento) más la acción que ejecutó, y
   aprende a *imaginar* qué van a sensar sus sensores en *t+1*.
2. **Sorpresa** — el error de esa predicción (sensor real vs. sensor imaginado).
   Si el modelo predictivo ya sabía lo que iba a pasar, sorpresa ≈ 0 (aburrido,
   predecible). Si no lo vio venir, sorpresa ≈ 1 (novedoso, informativo).
3. **Modelo Inverso (política de acción)** — igual que antes (sensores+necesidades
   → acción), pero ahora cada ejemplo de entrenamiento pesa según su *Sorpresa*:
   las transiciones más sorprendentes empujan más el aprendizaje de la red.

`babling.c` ya registra todo lo necesario (sensores, necesidades y acción en *t*,
y sensores en *t+1*), así que **no se modifica**. Tampoco se tocan `Epuck_25.c` ni
`server_cerebro.py`: la sorpresa se usa solo aquí, fuera de línea, para entrenar
mejor la red que termina cargando `server_cerebro.py`.

In [20]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.utils.class_weight import compute_class_weight

RUTA_DATASET = "/content/dataset_babbling.csv"

# Columnas tal cual las escribe babling.c
COLS_SENSORES_T  = ['DistIzq_t', 'DistDer_t', 'LuzIzq_t', 'LuzDer_t', 'SueloIzq_t', 'SueloDer_t']
COLS_NECESIDADES = ['Bat_t', 'Aburr_t']
COLS_SENSORES_T1 = ['DistIzq_t1', 'DistDer_t1', 'LuzIzq_t1', 'LuzDer_t1', 'SueloIzq_t1', 'SueloDer_t1']
COLS_ESTADO      = COLS_SENSORES_T + COLS_NECESIDADES   # 8 columnas: lo que "siente" el robot

In [21]:
def cargar_experiencia():
    print("1. Cargando la experiencia del E-puck (balbuceo motor)...")
    df = pd.read_csv(RUTA_DATASET)
    print(f"   -> {len(df)} transiciones (Estado_t, Accion, Estado_t+1) registradas.")
    return df

In [22]:
def entrenar_modelo_predictivo(df):
    """
    MODELO PREDICTIVO (Forward Model)
    Entrada:  Estado_t (sensores + necesidades) + Accion_t (one-hot, 3 valores)
    Salida:   Sensores_t+1 (lo que el robot espera sensar después de actuar)

    El error de esta red ES la "sorpresa": cuánto se equivoca al imaginar el
    futuro. Se entrena con TODAS las transiciones del balbuceo (no solo las
    "útiles"), porque necesita conocer la dinámica del mundo en general, buena
    o mala, para poder anticiparla.
    """
    print("\n2. Entrenando el MODELO PREDICTIVO (anticipa lo que se va a sensar)...")

    accion_onehot = pd.get_dummies(pd.Categorical(df["Accion"], categories=[0, 1, 2]), prefix="Accion")

    X_cont = df[COLS_ESTADO]
    Y = df[COLS_SENSORES_T1]

    scaler_fwd_X = StandardScaler().fit(X_cont)
    scaler_fwd_Y = StandardScaler().fit(Y)

    X_cont_scaled = scaler_fwd_X.transform(X_cont)
    Y_scaled = scaler_fwd_Y.transform(Y)
    X = np.hstack([X_cont_scaled, accion_onehot.values.astype(float)])  # 8 + 3 = 11 entradas

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y_scaled, test_size=0.2, random_state=42)

    modelo_predictivo = Sequential([
        Dense(64, activation="relu", input_shape=(11,)),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(6, activation="linear")   # Regresion: predice los 6 sensores futuros
    ])

    modelo_predictivo.compile(optimizer="adam", loss="mse", metrics=["mae"])

    parada_temprana = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

    modelo_predictivo.fit(
        X_train, Y_train,
        validation_data=(X_test, Y_test),
        epochs=150,
        batch_size=16,
        callbacks=[parada_temprana],
        verbose=1
    )

    modelo_predictivo.save_weights("pesos_forward_epuck.weights.h5")
    joblib.dump(scaler_fwd_X, "scaler_forward_X_epuck.pkl")
    joblib.dump(scaler_fwd_Y, "scaler_forward_Y_epuck.pkl")
    print("   -> Pesos y scalers del modelo predictivo guardados.")

    return modelo_predictivo, scaler_fwd_X, scaler_fwd_Y

In [23]:
def calcular_sorpresa(df, modelo_predictivo, scaler_fwd_X, scaler_fwd_Y):
    """
    Usa el modelo predictivo YA ENTRENADO para "imaginar" cada transicion del
    dataset completo y mide que tan lejos estuvo esa imaginacion de lo que
    realmente paso. Ese error es la SORPRESA: lo que mas le ensenia al robot.
    """
    print("\n3. Calculando la sorpresa (error de prediccion) de cada experiencia...")

    accion_onehot = pd.get_dummies(pd.Categorical(df["Accion"], categories=[0, 1, 2]), prefix="Accion")
    X_cont_scaled = scaler_fwd_X.transform(df[COLS_ESTADO])
    X = np.hstack([X_cont_scaled, accion_onehot.values.astype(float)])

    Y_pred_scaled = modelo_predictivo.predict(X, verbose=0)
    Y_pred = scaler_fwd_Y.inverse_transform(Y_pred_scaled)
    Y_real = df[COLS_SENSORES_T1].values

    error_cuadratico = np.mean((Y_pred - Y_real) ** 2, axis=1)

    # Normalizamos 0-1 y dejamos un piso de 0.1 para que ninguna muestra pese "cero"
    e_min, e_max = error_cuadratico.min(), error_cuadratico.max()
    sorpresa_norm = (error_cuadratico - e_min) / (e_max - e_min + 1e-9)

    df = df.copy()
    df["Sorpresa"] = sorpresa_norm
    df["Peso_Curiosidad"] = 0.1 + 0.9 * sorpresa_norm

    print(f"   -> Sorpresa promedio: {sorpresa_norm.mean():.3f} "
          f"(0 = totalmente predecible, 1 = totalmente inesperado)")
    return df

In [24]:
def entrenar_modelo_inverso(df):
    """
    MODELO INVERSO (Politica de accion)
    Entrada:  Estado_t (sensores + necesidades)
    Salida:   Accion que conviene tomar (0: Avanzar, 1: Girar Der, 2: Girar Izq)

    Se entrena SOLO con transiciones "utiles" (evito un obstaculo o se acerco a
    la luz), igual que antes -- pero ahora cada muestra pesa segun su Sorpresa:
    las transiciones mas sorprendentes (las que el modelo predictivo no supo
    anticipar) empujan mas fuerte el aprendizaje de esta red.
    """
    print("\n4. Entrenando el MODELO INVERSO (decide que accion tomar), ponderado por sorpresa...")

    filtro_esquivar = (
        ((df["DistIzq_t"] > 100) | (df["DistDer_t"] > 100)) &   # había obstáculo
        ((df["DistIzq_t1"] < df["DistIzq_t"]) | (df["DistDer_t1"] < df["DistDer_t"]))
    )
    # Útil para comer: había luz (lata cerca) y se acercó
    filtro_comer = (
        ((df["LuzIzq_t"] > 500) | (df["LuzDer_t"] > 500)) &    # había luz visible
        ((df["LuzIzq_t1"] > df["LuzIzq_t"]) | (df["LuzDer_t1"] > df["LuzDer_t"]))  # se acercó
    )
    # Útil para aburrimiento: estaba en zona negra
    filtro_negro = (df["SueloIzq_t"] < 400) | (df["SueloDer_t"] < 400)

    df_util = df[filtro_esquivar | filtro_comer | filtro_negro]
    print(f"   -> De {len(df)} pasos aleatorios, se usaran {len(df_util)} pasos utiles para aprender.")

    X = df_util[COLS_ESTADO]
    y = df_util["Accion"]
    pesos = df_util["Peso_Curiosidad"]

    X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
        X, y, pesos, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # OJO: server_cerebro.py carga exactamente "scaler_epuck.pkl" (sin el "3").
    # El notebook original lo guardaba como "scaler_epuck3.pkl" -> no coincidian.
    # Aqui se corrige guardandolo con el nombre que el servidor realmente espera.
    joblib.dump(scaler, "scaler_epuck4.pkl")

    modelo_inverso = Sequential([
        Dense(64, activation="relu", input_shape=(8,)),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(3, activation="softmax")
    ])

    modelo_inverso.compile(optimizer="adam",
                           loss="sparse_categorical_crossentropy",
                           metrics=["accuracy"])

    clases = np.array([0, 1, 2])
    pesos_clase = compute_class_weight('balanced', classes=clases, y=y_train.values)
    dict_pesos_clase = dict(enumerate(pesos_clase))
    print(f"   -> Pesos de clase: {dict_pesos_clase}")

    # Combinar con los pesos de curiosidad (multiplicar ambos)
    # sample_weight ya tiene Peso_Curiosidad, ahora también incorporamos el balanceo
    pesos_combinados = w_train.values * np.array([dict_pesos_clase[a] for a in y_train.values])

    modelo_inverso.fit(
        X_train_scaled, y_train,
        sample_weight=pesos_combinados,   # ← antes era solo w_train
        validation_data=(X_test_scaled, y_test),  # ← quitar w_test aquí, Keras lo ignora en val
        epochs=50,
        batch_size=32,
        verbose=1
    )

    modelo_inverso.save_weights("pesos_epuck4.weights.h5")
    print("   -> Pesos y scaler del modelo inverso guardados (listos para server_cerebro.py).")

    return modelo_inverso

In [25]:
def preparar_y_entrenar():
    df = cargar_experiencia()
    modelo_predictivo, scaler_fwd_X, scaler_fwd_Y = entrenar_modelo_predictivo(df)
    df_con_sorpresa = calcular_sorpresa(df, modelo_predictivo, scaler_fwd_X, scaler_fwd_Y)
    entrenar_modelo_inverso(df_con_sorpresa)
    print("\n>> Listo: modelo predictivo + modelo inverso curiosidad-ponderado entrenados. <<")

if __name__ == "__main__":
    preparar_y_entrenar()

1. Cargando la experiencia del E-puck (balbuceo motor)...
   -> 4839 transiciones (Estado_t, Accion, Estado_t+1) registradas.

2. Entrenando el MODELO PREDICTIVO (anticipa lo que se va a sensar)...
Epoch 1/150


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


242/242 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/150
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/150
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/150
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/150
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/150
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/150
 30/242 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: nan - mae: nan

KeyboardInterrupt: 